In [4]:
# RUN 

In [ ]:
# ==============================
# 0_FNS – VIDEO URL → MP4 + GIF
#       + GIF COLOR → TRANSPARENCY
# ==============================

import os
import sys
import requests
from datetime import datetime

import numpy as np
import imageio.v2 as imageio

try:
    from moviepy.editor import VideoFileClip
except ImportError as e:
    print("\n[ERROR] moviepy not installed.")
    print("Install:\n   pip install moviepy imageio-ffmpeg\n")
    raise e


# -----######-----###### MAIN FUNCTION 1 -----######-----######
def _video_0912_url2gif_GET_paths(
    url,
    keyword="transition",
    start_sec=None,
    end_sec=None,
    resize_width=720,
    fps=12
):
    """
    Download MP4 from URL and export GIF into:
        ~/Desktop/__out_MM_DD_REP_(keyword)_

    Returns
    -------
    dict
        {
          "output_dir": <folder>,
          "mp4_path": <mp4>,
          "gif_path": <gif>
        }
    """

    # -------------------------
    # 1) Prepare output folder
    # -------------------------
    today = datetime.now().strftime("%m_%d")
    desktop = os.path.join(os.path.expanduser("~"), "Desktop")
    out_dir = os.path.join(desktop, f"__out_{today}_REP_{keyword}_")

    if not os.path.isdir(out_dir):
        os.makedirs(out_dir, exist_ok=True)

    base_name = f"republic_{keyword}"
    mp4_path = os.path.join(out_dir, f"{base_name}.mp4")
    gif_path = os.path.join(out_dir, f"{base_name}.gif")

    print("\n====================================")
    print("   VIDEO URL → MP4 + GIF PIPELINE   ")
    print("====================================")
    print(f"[INFO] URL          : {url}")
    print(f"[INFO] Keyword      : {keyword}")
    print(f"[INFO] Output Dir   : {out_dir}")
    print(f"[INFO] MP4 Path     : {mp4_path}")
    print(f"[INFO] GIF Path     : {gif_path}")
    print("------------------------------------")

    # -------------------------
    # 2) Download the MP4 (TQM-ish)
    # -------------------------
    print("[STEP 1/3] Downloading MP4...")

    try:
        with requests.get(url, stream=True) as r:
            r.raise_for_status()
            total = int(r.headers.get("content-length", 0))
            downloaded = 0

            with open(mp4_path, "wb") as f:
                for chunk in r.iter_content(chunk_size=8192):
                    if not chunk:
                        continue
                    f.write(chunk)
                    downloaded += len(chunk)

                    if total > 0:
                        pct = (downloaded / total) * 100
                        sys.stdout.write(
                            f"\r    Progress: [{pct:6.2f}%] {downloaded/1_000_000:7.2f} MB"
                        )
                        sys.stdout.flush()

        print("\n[OK] MP4 downloaded.")
    except Exception as e:
        print("\n[ERROR] Download failed:")
        print(e)
        raise

    # -------------------------
    # 3) Load and prepare clip
    # -------------------------
    print("[STEP 2/3] Processing video...")

    try:
        clip = VideoFileClip(mp4_path)

        if start_sec is not None or end_sec is not None:
            t0 = start_sec if start_sec is not None else 0
            t1 = end_sec if end_sec is not None else clip.duration
            t0 = max(0, float(t0))
            t1 = min(float(t1), clip.duration)
            if t1 <= t0:
                raise ValueError(f"Invalid segment: start={t0}, end={t1}")
            clip = clip.subclip(t0, t1)

        if resize_width:
            clip = clip.resize(width=resize_width)

        print(f"[INFO] GIF duration : {clip.duration:.2f} sec")
        print(f"[INFO] GIF fps      : {fps}")
        if resize_width:
            print(f"[INFO] GIF width    : {resize_width}px")

    except Exception as e:
        print("[ERROR] Failed processing video:")
        print(e)
        raise

    # -------------------------
    # 4) Export GIF
    # -------------------------
    print("[STEP 3/3] Exporting GIF...")

    try:
        clip.write_gif(gif_path, fps=fps)
        clip.close()
        print("[OK] GIF exported.")
    except Exception as e:
        print("[ERROR] GIF export failed:")
        print(e)
        raise

    print("------------------------------------")
    print("[DONE] Files saved in:")
    print(out_dir)
    print("====================================\n")

    return {
        "output_dir": out_dir,
        "mp4_path": mp4_path,
        "gif_path": gif_path
    }


# -----######-----###### MAIN FUNCTION 2 (TOLERANCE) -----######-----######
def _gif_0912_color2alpha_GET_path(gif_path, mode="none", tol=40):
    """
    Take a GIF and make one color transparent, with tolerance.

    Parameters
    ----------
    gif_path : str
        Path to the input GIF.
    mode : str
        "black", "white", or "none".
    tol : int
        Tolerance for how close a pixel is to black/white (0–255 range).
        - For "black": pixels with avg brightness <= tol become transparent.
        - For "white": pixels with avg brightness >= 255 - tol become transparent.

    Returns
    -------
    str
        Path to the new transparent GIF (or original if mode == "none").
    """

    if not gif_path or not os.path.isfile(gif_path):
        raise FileNotFoundError(f"GIF not found: {gif_path}")

    mode = (mode or "none").strip().lower()
    if mode not in {"black", "white", "none"}:
        mode = "none"

    if mode == "none":
        print("\n[INFO] Transparency mode = none. Skipping GIF alpha processing.")
        return gif_path

    tol = max(0, min(int(tol), 255))  # clamp

    print("\n====================================")
    print("       GIF COLOR → TRANSPARENCY     ")
    print("====================================")
    print(f"[INFO] Input GIF  : {gif_path}")
    print(f"[INFO] Mode       : {mode}")
    print(f"[INFO] Tolerance  : {tol}")
    print("------------------------------------")

    reader = imageio.get_reader(gif_path)
    meta = reader.get_meta_data()
    duration = meta.get("duration", 0.08)  # seconds per frame (fallback)

    frames_out = []
    total_frames = reader.get_length() if hasattr(reader, "get_length") else None

    print("[STEP] Processing frames and applying transparency...")

    for i, frame in enumerate(reader):
        arr = frame

        # Ensure we have at least RGB
        if arr.ndim == 2:
            arr = np.stack([arr] * 3, axis=-1)

        # Ensure RGBA
        if arr.shape[2] == 3:
            alpha = np.full(arr.shape[:2] + (1,), 255, dtype=np.uint8)
            arr = np.concatenate([arr, alpha], axis=-1)

        r = arr[..., 0].astype(np.int16)
        g = arr[..., 1].astype(np.int16)
        b = arr[..., 2].astype(np.int16)
        a = arr[..., 3]

        # Average brightness per pixel
        brightness = (r + g + b) / 3.0

        if mode == "black":
            mask = brightness <= tol
        else:  # "white"
            mask = brightness >= (255 - tol)

        a[mask] = 0
        arr[..., 3] = a

        frames_out.append(arr.astype(np.uint8))

        # TQM-ish bar
        if total_frames:
            pct = (i + 1) / total_frames * 100
            sys.stdout.write(f"\r    Frames: {i+1}/{total_frames} [{pct:6.2f}%]")
        else:
            sys.stdout.write(f"\r    Frames processed: {i+1}")
        sys.stdout.flush()

    reader.close()
    print("\n[STEP] Saving transparent GIF...")

    folder, name = os.path.split(gif_path)
    name_no_ext, _ = os.path.splitext(name)
    out_path = os.path.join(folder, f"{name_no_ext}_transp.gif")

    imageio.mimsave(out_path, frames_out, duration=duration, loop=0)
    print("[OK] Transparent GIF saved:")
    print(out_path)
    print("====================================\n")

    return out_path


# !!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
#!#!#!#!#! RUNNING STATEMENTS #!#!#!#!#!
# !!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!

if __name__ == "__main__":
    # 1) Ask for MP4 URL
    url_in = input("\nPaste the MP4 URL:\n> ").strip()
    if not url_in:
        raise ValueError("No URL provided. Exiting.")

    # 2) Ask keyword for output folder name
    keyword_in = input("Keyword for output folder name (e.g. 'water'):\n> ").strip()
    if not keyword_in:
        keyword_in = "transition"

    # 3) Run URL → MP4 + GIF
    paths = _video_0912_url2gif_GET_paths(
        url=url_in,
        keyword=keyword_in,
        start_sec=None,      # or e.g. 0
        end_sec=None,        # or e.g. 5
        resize_width=720,    # set None to keep original width
        fps=12
    )

    print("Returned paths:")
    print(paths)

    # 4) Ask user for GIF path to post-process
    default_gif = paths.get("gif_path")
    print("\nGIF created:")
    print(f"  {default_gif}")

    gif_path_in = input(
        "\nPath of GIF to apply transparency "
        f"[press Enter to use above]:\n> "
    ).strip()

    if not gif_path_in:
        gif_path_in = default_gif

    # 5) Ask transparency mode
    print("\nChoose color to make transparent in GIF:")
    print("  1 = black")
    print("  2 = white")
    print("  3 = none (no transparency)")
    choice = input("> ").strip()

    if choice == "1":
        mode = "black"
    elif choice == "2":
        mode = "white"
        # mode = "white"
    else:
        mode = "none"

    # If black edges remain, bump tol (e.g. 50–60)
    out_gif_transp = _gif_0912_color2alpha_GET_path(
        gif_path_in,
        mode=mode,
        tol=40
    )

    print("\nFinal GIF path:")
    print(out_gif_transp)



Paste the MP4 URL:
>  https://storage.googleapis.com/republiclabs/ygvargas93@gmail.com/gnpxf9db2srmw0cvntp832b6nm.mp4
Keyword for output folder name (e.g. 'water'):
>  logo5



   VIDEO URL → MP4 + GIF PIPELINE   
[INFO] URL          : https://storage.googleapis.com/republiclabs/ygvargas93@gmail.com/gnpxf9db2srmw0cvntp832b6nm.mp4
[INFO] Keyword      : logo5
[INFO] Output Dir   : /Users/yerik/Desktop/__out_01_11_REP_logo5_
[INFO] MP4 Path     : /Users/yerik/Desktop/__out_01_11_REP_logo5_/republic_logo5.mp4
[INFO] GIF Path     : /Users/yerik/Desktop/__out_01_11_REP_logo5_/republic_logo5.gif
------------------------------------
[STEP 1/3] Downloading MP4...
    Progress: [100.00%]    0.71 MB
[OK] MP4 downloaded.
[STEP 2/3] Processing video...
[INFO] GIF duration : 5.06 sec
[INFO] GIF fps      : 12
[INFO] GIF width    : 720px
[STEP 3/3] Exporting GIF...
MoviePy - Building file /Users/yerik/Desktop/__out_01_11_REP_logo5_/republic_logo5.gif with imageio.


[OK] GIF exported.
------------------------------------
[DONE] Files saved in:
/Users/yerik/Desktop/__out_01_11_REP_logo5_

Returned paths:
{'output_dir': '/Users/yerik/Desktop/__out_01_11_REP_logo5_', 'mp4_path': '/Users/yerik/Desktop/__out_01_11_REP_logo5_/republic_logo5.mp4', 'gif_path': '/Users/yerik/Desktop/__out_01_11_REP_logo5_/republic_logo5.gif'}

GIF created:
  /Users/yerik/Desktop/__out_01_11_REP_logo5_/republic_logo5.gif


In [5]:
# ==============================
# 0_FNS – VIDEO URL → MP4 + GIF
# ==============================

import os
import sys
import requests
from datetime import datetime

import numpy as np
import imageio.v2 as imageio

try:
    from moviepy.editor import VideoFileClip
except ImportError as e:
    print("\n[ERROR] moviepy not installed.")
    print("Install:\n   pip install moviepy imageio-ffmpeg\n")
    raise e


# -----######-----###### MAIN FUNCTION 1 -----######-----######
def _video_0912_url2gif_GET_paths(
    url,
    keyword="transition",
    start_sec=None,
    end_sec=None,
    resize_width=720,
    fps=12
):
    """
    Download MP4 from URL and export GIF into:
        ~/Desktop/__out_MM_DD_REP_(keyword)_

    Returns
    -------
    dict
        {
          "output_dir": <folder>,
          "mp4_path": <mp4>,
          "gif_path": <gif>
        }
    """

    # -------------------------
    # 1) Prepare output folder
    # -------------------------
    today = datetime.now().strftime("%m_%d")
    desktop = os.path.join(os.path.expanduser("~"), "Desktop")
    out_dir = os.path.join(desktop, f"__out_{today}_REP_{keyword}_")

    if not os.path.isdir(out_dir):
        os.makedirs(out_dir, exist_ok=True)

    base_name = f"republic_{keyword}"
    mp4_path = os.path.join(out_dir, f"{base_name}.mp4")
    gif_path = os.path.join(out_dir, f"{base_name}.gif")

    print("\n====================================")
    print("   VIDEO URL → MP4 + GIF PIPELINE   ")
    print("====================================")
    print(f"[INFO] URL          : {url}")
    print(f"[INFO] Keyword      : {keyword}")
    print(f"[INFO] Output Dir   : {out_dir}")
    print(f"[INFO] MP4 Path     : {mp4_path}")
    print(f"[INFO] GIF Path     : {gif_path}")
    print("------------------------------------")

    # -------------------------
    # 2) Download the MP4 (TQM-ish)
    # -------------------------
    print("[STEP 1/3] Downloading MP4...")

    try:
        with requests.get(url, stream=True) as r:
            r.raise_for_status()
            total = int(r.headers.get("content-length", 0))
            downloaded = 0

            with open(mp4_path, "wb") as f:
                for chunk in r.iter_content(chunk_size=8192):
                    if not chunk:
                        continue
                    f.write(chunk)
                    downloaded += len(chunk)

                    if total > 0:
                        pct = (downloaded / total) * 100
                        sys.stdout.write(
                            f"\r    Progress: [{pct:6.2f}%] {downloaded/1_000_000:7.2f} MB"
                        )
                        sys.stdout.flush()

        print("\n[OK] MP4 downloaded.")
    except Exception as e:
        print("\n[ERROR] Download failed:")
        print(e)
        raise

    # -------------------------
    # 3) Load and prepare clip
    # -------------------------
    print("[STEP 2/3] Processing video...")

    try:
        clip = VideoFileClip(mp4_path)

        if start_sec is not None or end_sec is not None:
            t0 = start_sec if start_sec is not None else 0
            t1 = end_sec if end_sec is not None else clip.duration
            t0 = max(0, float(t0))
            t1 = min(float(t1), clip.duration)
            if t1 <= t0:
                raise ValueError(f"Invalid segment: start={t0}, end={t1}")
            clip = clip.subclip(t0, t1)

        if resize_width:
            clip = clip.resize(width=resize_width)

        print(f"[INFO] GIF duration : {clip.duration:.2f} sec")
        print(f"[INFO] GIF fps      : {fps}")
        if resize_width:
            print(f"[INFO] GIF width    : {resize_width}px")

    except Exception as e:
        print("[ERROR] Failed processing video:")
        print(e)
        raise

    # -------------------------
    # 4) Export GIF
    # -------------------------
    print("[STEP 3/3] Exporting GIF...")

    try:
        clip.write_gif(gif_path, fps=fps)
        clip.close()
        print("[OK] GIF exported.")
    except Exception as e:
        print("[ERROR] GIF export failed:")
        print(e)
        raise

    print("------------------------------------")
    print("[DONE] Files saved in:")
    print(out_dir)
    print("====================================\n")

    return {
        "output_dir": out_dir,
        "mp4_path": mp4_path,
        "gif_path": gif_path
    }


# -----######-----###### MAIN FUNCTION 2 -----######-----######
def _gif_0912_color2alpha_GET_path(gif_path, mode="none"):
    """
    Take a GIF and make one color transparent.

    Parameters
    ----------
    gif_path : str
        Path to the input GIF.
    mode : str
        "black", "white", or "none".

    Returns
    -------
    str
        Path to the new transparent GIF (or original if mode == "none").
    """

    if not gif_path or not os.path.isfile(gif_path):
        raise FileNotFoundError(f"GIF not found: {gif_path}")

    mode = (mode or "none").strip().lower()
    if mode not in {"black", "white", "none"}:
        mode = "none"

    if mode == "none":
        print("\n[INFO] Transparency mode = none. Skipping GIF alpha processing.")
        return gif_path

    target_color = (0, 0, 0) if mode == "black" else (255, 255, 255)

    folder, name = os.path.split(gif_path)
    name_no_ext, _ = os.path.splitext(name)
    out_path = os.path.join(folder, f"{name_no_ext}_transp.gif")

    print("\n====================================")
    print("       GIF COLOR → TRANSPARENCY     ")
    print("====================================")
    print(f"[INFO] Input GIF  : {gif_path}")
    print(f"[INFO] Output GIF : {out_path}")
    print(f"[INFO] Mode       : {mode} → {target_color}")
    print("------------------------------------")

    reader = imageio.get_reader(gif_path)
    meta = reader.get_meta_data()
    duration = meta.get("duration", 0.08)  # seconds per frame (fallback)

    frames_out = []
    total_frames = reader.get_length() if hasattr(reader, "get_length") else None

    print("[STEP] Processing frames and applying transparency...")

    for i, frame in enumerate(reader):
        arr = frame
        if arr.ndim == 2:
            # grayscale → RGB
            arr = np.stack([arr]*3, axis=-1)

        if arr.shape[2] == 3:
            # add full alpha channel
            alpha = np.full(arr.shape[:2] + (1,), 255, dtype=np.uint8)
            arr = np.concatenate([arr, alpha], axis=-1)

        r, g, b, a = arr[..., 0], arr[..., 1], arr[..., 2], arr[..., 3]

        mask = (r == target_color[0]) & (g == target_color[1]) & (b == target_color[2])
        a[mask] = 0
        arr[..., 3] = a

        frames_out.append(arr)

        # TQM-ish bar
        if total_frames:
            pct = (i + 1) / total_frames * 100
            sys.stdout.write(f"\r    Frames: {i+1}/{total_frames} [{pct:6.2f}%]")
        else:
            sys.stdout.write(f"\r    Frames processed: {i+1}")
        sys.stdout.flush()

    reader.close()
    print("\n[STEP] Saving transparent GIF...")

    imageio.mimsave(out_path, frames_out, duration=duration, loop=0)
    print("[OK] Transparent GIF saved.")
    print("====================================\n")

    return out_path
#!#!#!#!#! RUNNING STATEMENTS #!#!#!#!#!

if __name__ == "__main__":
    # 1) Ask for MP4 URL
    url_in = input("\nPaste the MP4 URL:\n> ").strip()
    if not url_in:
        raise ValueError("No URL provided. Exiting.")

    # 2) Ask keyword for output folder name
    keyword_in = input("Keyword for output folder name (e.g. 'water'):\n> ").strip()
    if not keyword_in:
        keyword_in = "transition"

    # 3) Run URL → MP4 + GIF
    paths = _video_0912_url2gif_GET_paths(
        url=url_in,
        keyword=keyword_in,
        start_sec=None,      # or e.g. 0
        end_sec=None,        # or e.g. 5
        resize_width=720,    # set None to keep original width
        fps=12
    )

    print("Returned paths:")
    print(paths)

    # 4) Ask user for GIF path to post-process
    default_gif = paths.get("gif_path")
    print("\nGIF created:")
    print(f"  {default_gif}")

    gif_path_in = input(
        "\nPath of GIF to apply transparency "
        f"[press Enter to use above]:\n> "
    ).strip()

    if not gif_path_in:
        gif_path_in = default_gif

    # 5) Ask transparency mode
    print("\nChoose color to make transparent in GIF:")
    print("  1 = black")
    print("  2 = white")
    print("  3 = none (no transparency)")
    choice = input("> ").strip()

    if choice == "1":
        mode = "black"
    elif choice == "2":
        mode = "white"
    else:
        mode = "none"

    out_gif_transp = _gif_0912_color2alpha_GET_path(gif_path_in, mode=mode)

    print("\nFinal GIF path:")
    print(out_gif_transp)



Paste the MP4 URL:
>  https://storage.googleapis.com/republiclabs/ygvargas93@gmail.com/hqy07k24thrma0cv0kzbqvzs6c.mp4
Keyword for output folder name (e.g. 'water'):
>  logo



   VIDEO URL → MP4 + GIF PIPELINE   
[INFO] URL          : https://storage.googleapis.com/republiclabs/ygvargas93@gmail.com/hqy07k24thrma0cv0kzbqvzs6c.mp4
[INFO] Keyword      : logo
[INFO] Output Dir   : /Users/yerik/Desktop/__out_12_09_REP_logo_
[INFO] MP4 Path     : /Users/yerik/Desktop/__out_12_09_REP_logo_/republic_logo.mp4
[INFO] GIF Path     : /Users/yerik/Desktop/__out_12_09_REP_logo_/republic_logo.gif
------------------------------------
[STEP 1/3] Downloading MP4...
    Progress: [100.00%]    7.54 MB
[OK] MP4 downloaded.
[STEP 2/3] Processing video...
[INFO] GIF duration : 5.04 sec
[INFO] GIF fps      : 12
[INFO] GIF width    : 720px
[STEP 3/3] Exporting GIF...
MoviePy - Building file /Users/yerik/Desktop/__out_12_09_REP_logo_/republic_logo.gif with imageio.


[OK] GIF exported.
------------------------------------
[DONE] Files saved in:
/Users/yerik/Desktop/__out_12_09_REP_logo_

Returned paths:
{'output_dir': '/Users/yerik/Desktop/__out_12_09_REP_logo_', 'mp4_path': '/Users/yerik/Desktop/__out_12_09_REP_logo_/republic_logo.mp4', 'gif_path': '/Users/yerik/Desktop/__out_12_09_REP_logo_/republic_logo.gif'}

GIF created:
  /Users/yerik/Desktop/__out_12_09_REP_logo_/republic_logo.gif



Path of GIF to apply transparency [press Enter to use above]:
>  



Choose color to make transparent in GIF:
  1 = black
  2 = white
  3 = none (no transparency)


>  1



       GIF COLOR → TRANSPARENCY     
[INFO] Input GIF  : /Users/yerik/Desktop/__out_12_09_REP_logo_/republic_logo.gif
[INFO] Output GIF : /Users/yerik/Desktop/__out_12_09_REP_logo_/republic_logo_transp.gif
[INFO] Mode       : black → (0, 0, 0)
------------------------------------
[STEP] Processing frames and applying transparency...
    Frames: 61/61 [100.00%]
[STEP] Saving transparent GIF...
[OK] Transparent GIF saved.


Final GIF path:
/Users/yerik/Desktop/__out_12_09_REP_logo_/republic_logo_transp.gif


In [3]:
https://storage.googleapis.com/republiclabs/ygvargas93@gmail.com/hqy07k24thrma0cv0kzbqvzs6c.mp4

SyntaxError: invalid syntax (2637994784.py, line 1)

In [1]:
# ==============================
# 0_FNS – VIDEO URL → MP4 + GIF
# ==============================

import os
import sys
import requests
from datetime import datetime

try:
    from moviepy.editor import VideoFileClip
except ImportError as e:
    print("\n[ERROR] moviepy not installed.")
    print("Install:\n   pip install moviepy imageio-ffmpeg\n")
    raise e


# -----######-----###### MAIN FUNCTION -----######-----######
def _video_0912_url2gif_GET_paths(
    url,
    keyword="transition",
    start_sec=None,
    end_sec=None,
    resize_width=720,
    fps=12
):
    """
    Downloads MP4 from URL and exports GIF into:
        ~/Desktop/__out_MM_DD_REP_(keyword)_
    """

    # -------------------------
    # 1) Prepare output folder
    # -------------------------
    today = datetime.now().strftime("%m_%d")
    desktop = os.path.join(os.path.expanduser("~"), "Desktop")

    out_dir = os.path.join(desktop, f"__out_{today}_REP_{keyword}_")

    if not os.path.isdir(out_dir):
        os.makedirs(out_dir, exist_ok=True)

    base_name = f"republic_{keyword}"
    mp4_path = os.path.join(out_dir, f"{base_name}.mp4")
    gif_path = os.path.join(out_dir, f"{base_name}.gif")

    print("\n====================================")
    print("   VIDEO URL → MP4 + GIF PIPELINE   ")
    print("====================================")
    print(f"[INFO] URL          : {url}")
    print(f"[INFO] Keyword      : {keyword}")
    print(f"[INFO] Output Dir   : {out_dir}")
    print(f"[INFO] MP4 Path     : {mp4_path}")
    print(f"[INFO] GIF Path     : {gif_path}")
    print("------------------------------------")

    # -------------------------
    # 2) Download the MP4
    # -------------------------
    print("[STEP 1/3] Downloading MP4...")

    try:
        with requests.get(url, stream=True) as r:
            r.raise_for_status()
            total = int(r.headers.get("content-length", 0))
            downloaded = 0

            with open(mp4_path, "wb") as f:
                for chunk in r.iter_content(chunk_size=8192):
                    if not chunk:
                        continue
                    f.write(chunk)
                    downloaded += len(chunk)

                    if total > 0:
                        pct = (downloaded / total) * 100
                        sys.stdout.write(
                            f"\r    Progress: [{pct:6.2f}%] {downloaded/1_000_000:7.2f} MB"
                        )
                        sys.stdout.flush()

        print("\n[OK] MP4 downloaded.")
    except Exception as e:
        print("\n[ERROR] Download failed:")
        print(e)
        raise

    # -------------------------
    # 3) Load and prepare clip
    # -------------------------
    print("[STEP 2/3] Processing video...")

    try:
        clip = VideoFileClip(mp4_path)

        if start_sec is not None or end_sec is not None:
            t0 = start_sec if start_sec is not None else 0
            t1 = end_sec if end_sec is not None else clip.duration
            clip = clip.subclip(t0, t1)

        if resize_width:
            clip = clip.resize(width=resize_width)

    except Exception as e:
        print("[ERROR] Failed processing video:")
        print(e)
        raise

    # -------------------------
    # 4) Export GIF
    # -------------------------
    print("[STEP 3/3] Exporting GIF...")

    try:
        clip.write_gif(gif_path, fps=fps)
        clip.close()
        print("[OK] GIF exported.")
    except Exception as e:
        print("[ERROR] GIF export failed:")
        print(e)
        raise

    print("------------------------------------")
    print("[DONE] Files saved in:")
    print(out_dir)
    print("====================================\n")

    return {
        "output_dir": out_dir,
        "mp4_path": mp4_path,
        "gif_path": gif_path
    }


In [2]:
#!#!#!#!#! RUNNING STATEMENTS #!#!#!#!#!

if __name__ == "__main__":
    url_in = input("\nPaste the MP4 URL:\n> ").strip()
    keyword_in = input("Keyword for output folder name:\n> ").strip()

    result = _video_0912_url2gif_GET_paths(
        url=url_in,
        keyword=keyword_in,
        start_sec=None,
        end_sec=None,
        resize_width=720,
        fps=12
    )

    print("\nReturned paths:")
    print(result)



Paste the MP4 URL:
>  https://storage.googleapis.com/republiclabs/ygvargas93@gmail.com/hqy07k24thrma0cv0kzbqvzs6c.mp4
Keyword for output folder name:
>  MoM_logo1



   VIDEO URL → MP4 + GIF PIPELINE   
[INFO] URL          : https://storage.googleapis.com/republiclabs/ygvargas93@gmail.com/hqy07k24thrma0cv0kzbqvzs6c.mp4
[INFO] Keyword      : MoM_logo1
[INFO] Output Dir   : /Users/yerik/Desktop/__out_12_09_REP_MoM_logo1_
[INFO] MP4 Path     : /Users/yerik/Desktop/__out_12_09_REP_MoM_logo1_/republic_MoM_logo1.mp4
[INFO] GIF Path     : /Users/yerik/Desktop/__out_12_09_REP_MoM_logo1_/republic_MoM_logo1.gif
------------------------------------
[STEP 1/3] Downloading MP4...
    Progress: [100.00%]    7.54 MB
[OK] MP4 downloaded.
[STEP 2/3] Processing video...
[STEP 3/3] Exporting GIF...
MoviePy - Building file /Users/yerik/Desktop/__out_12_09_REP_MoM_logo1_/republic_MoM_logo1.gif with imageio.


[OK] GIF exported.
------------------------------------
[DONE] Files saved in:
/Users/yerik/Desktop/__out_12_09_REP_MoM_logo1_


Returned paths:
{'output_dir': '/Users/yerik/Desktop/__out_12_09_REP_MoM_logo1_', 'mp4_path': '/Users/yerik/Desktop/__out_12_09_REP_MoM_logo1_/republic_MoM_logo1.mp4', 'gif_path': '/Users/yerik/Desktop/__out_12_09_REP_MoM_logo1_/republic_MoM_logo1.gif'}
